# First simple approach with a 2d cnn

In [ ]:
import pandas as pd
import torch
import torchvision
import torchvision.transforms.v2 as transforms_v2
import pydicom
import numpy as np
import os
from pathlib import Path
import optuna
import metric
# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Visualize a sample DICOM image
import matplotlib.pyplot as plt

dcm = '/kaggle/input/rsna-intracranial-aneurysm-detection/series/1.2.826.0.1.3680043.8.498.10046318991957083423208748012349179640/1.2.826.0.1.3680043.8.498.11550049804049536118256417115804980060.dcm'

ds = pydicom.dcmread(dcm)
plt.imshow(ds.pixel_array)
plt.show()

In [ ]:
from concurrent.futures import ThreadPoolExecutor
# Function to find 3D DICOM files in a series directory
if False:
    def find_3d_dicoms_in_series(series_dir):
        found = []
        for file in os.listdir(series_dir):
            if file.endswith('.dcm'):
                dcm_path = Path(series_dir) / file
                ds = pydicom.dcmread(dcm_path)
                if ds.pixel_array.ndim >= 3:
                    # print(f"3D DICOM found: {dcm_path}")
                    found.append(dcm_path)
        return found

    series_root = '/kaggle/input/rsna-intracranial-aneurysm-detection/series'
    series_dirs = [os.path.join(series_root, d) for d in os.listdir(series_root) if os.path.isdir(os.path.join(series_root, d))]

    mlt_frame_dcm_files = []
    with ThreadPoolExecutor() as executor:
        results = executor.map(find_3d_dicoms_in_series, series_dirs)
        for r in results:
            mlt_frame_dcm_files.extend(r)
            
    # save the list of DICOM files to a CSV file
    mlt_frame_dcm_files_df = pd.DataFrame(mlt_frame_dcm_files, columns=['dicom_path'])
    mlt_frame_dcm_files_df.to_csv('mlt_frame_dcm_files_2.csv', index=False)

In [ ]:
#Data preparation
root_path = '/kaggle/input/rsna-intracranial-aneurysm-detection'

train_df = pd.read_csv(os.path.join(root_path,'train.csv'))
train_loc_df = pd.read_csv(os.path.join(root_path,'train_localizers.csv'))
print(train_loc_df.columns)
multi_dicom_df = pd.read_csv('/kaggle/input/mlt-frame-dcm/mlt_frame_dcm_files_2.csv') # multi-frame/unexpected format DICOMs

series_path = os.path.join(root_path,'series')
classification_df = dict()
index_map = [] # to be able to use a map style dataset

break_after = 0 # for testing, remove later

for _, row in train_df.iterrows():
    series_id = row.iloc[0]
    aneurysm = row.iloc[-14:] # target
    
    for dcm_id in os.listdir(os.path.join(series_path, series_id)):
        # Skip multi-frame/unexpected DICOMs
        if os.path.join(series_path, series_id, dcm_id) in multi_dicom_df.values:
            #print(f"Skipping multi-frame DICOM: {dcm_id} in series {series_id}")
            continue
        dcm_id = Path(dcm_id).stem
        classification_df[(series_id, dcm_id)] = np.zeros(14, dtype=int)
        index_map.append((series_id, dcm_id))
    # set correct target for dcms in train_localizers
    if aneurysm.iloc[-1] == 1:
        break_after += 1
        for _, loc_row in train_loc_df[train_loc_df.iloc[:,0]==series_id].iterrows():
            if (series_id, loc_row.iloc[1]) in classification_df.keys():
                classification_df[(series_id, loc_row.iloc[1])] = aneurysm.values
    if break_after > 10: break
            
assert len(classification_df.keys()) == len(index_map)

print(type(list(classification_df.values())[1][0]))

# get indices of pos and neg samples
positive_indices = [i for i, (series_id, dcm_id) in enumerate(index_map) if classification_df[(series_id, dcm_id)][-1] == 1]
negative_indices = [i for i, (series_id, dcm_id) in enumerate(index_map) if classification_df[(series_id, dcm_id)][-1] == 0]
np.random.shuffle(positive_indices)
np.random.shuffle(negative_indices)
# Build validation set with balanced classes
val_positive_indices = positive_indices[:int(0.2 * len(positive_indices))]
val_negative_indices = negative_indices[:int(0.2 * len(positive_indices))]
val_indices = val_positive_indices + val_negative_indices
# Build training set with remaining samples
train_indices = [i for i in range(len(index_map)) if i not in val_indices]

print("Positive:", len(positive_indices), "Negative:", len(negative_indices))

In [ ]:
# TODO: Transforms 
train_transforms_1 = transforms_v2.Compose([
    transforms_v2.Resize((512, 512)),
    transforms_v2.RandomRotation(degrees=15),
    transforms_v2.RandomHorizontalFlip(),
    transforms_v2.ColorJitter(brightness=0.2, contrast=0.2),
])
val_transforms = transforms_v2.Compose([
    transforms_v2.Resize((512, 512)),
])

class AneurysmDataset(torch.utils.data.Dataset):
    """Dataset for RSNA Intracranial Aneurysm Detection.
    Args:
        root_path (str): Path to the dataset root directory.
        train (bool): If True, use the training set; if False, use the validation set.
    """
    def __init__(self, root_path, train=True, transforms=None):
        super().__init__()
        self.root_path = root_path
        self.classification_df = classification_df
        self.train = train
        self.transforms = transforms
        # create index map based on train or validation indices
        if train:
            self.index_map = {new_i: index_map[idx] for new_i, idx in enumerate(train_indices)}
        else:
            self.index_map = {new_i: index_map[idx] for new_i, idx in enumerate(val_indices)}

    def __getitem__(self, index):
        img_id = self.index_map[index] 
        img_path = os.path.join(self.root_path, 'series', img_id[0], img_id[1]+'.dcm')
        target = torch.tensor(self.classification_df[img_id].astype(int), dtype=torch.float, device=DEVICE)
        img = torch.tensor(pydicom.dcmread(img_path).pixel_array, dtype=torch.float, device=DEVICE).unsqueeze(0) # add channel dimension
        if self.transforms is not None: img = self.transforms(img)
        return img, target

    def __len__(self):
        return len(self.index_map.keys())
    
def make_train_loader(root_path, alpha=2, batch_size=32):
    """ Create a DataLoader for the training set with balanced classes.
    Args:
        root_path (str): Path to the dataset root directory.
        alpha (float): Weighting factor for the negative class sampling.
        The higher the value, the more negative samples are included in each epoch.
        batch_size (int): Size of the batches to be returned by the DataLoader.
    Returns:
        torch.utils.data.DataLoader: DataLoader for the training set with balanced classes.
    """
    dataset = AneurysmDataset(root_path, train=True, transforms=train_transforms_1)
    # sampler to ensure balanced classes in each batch
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=[1.0 / len(positive_indices) if dataset.classification_df[dataset.index_map[i]][-1] == 1 else alpha / len(negative_indices) for i in range(len(dataset))],
        num_samples=len(positive_indices)*3, # reduce dataset size to avoid too much resampling
        replacement=True
    )
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler)

## transforms thoughts
the images are 1 channel with deferrent modality (CT, MRI, etc). So nees to lookup in the dicom metadata what are the relevant info on scaling of pixel values, etc.  
Multimodal means that each mod should probably vary a lot in __contarst__ and __brightness__. So one model can procees all types. 
__Crop__ the images with attention to where the aneurysm is, make sure that the label is correct if the aneurysm is outside the crop.  
Apply __random rotation__ and __random flip__ augmentations.
Make variations on the strength of augmentations and augmentation selection in one trainning transform to have it as a variable in the optuna study.

In [ ]:
if False:
    dataset = AneurysmDataset('/kaggle/input/rsna-intracranial-aneurysm-detection')
    print(dataset[1][0].size())
    len(dataset)
    dl = torch.utils.data.DataLoader(dataset,12)
    for i, l in dl:
        print(i, l)

In [ ]:
#Train loop

def train_one_epoch(model, dataloader, optimizer, loss_fn):
    model.train()
    for images, labels in dataloader:
        preds = model(images)
        loss = loss_fn(preds,labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
def test_model(model, dataloader, loss_fn):
    model.eval()
    all_preds = torch.empty(0, device=DEVICE)
    all_labels = torch.empty(0, device=DEVICE)

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)  # ensure same device
            preds = model(images)
            all_preds = torch.cat((all_preds, preds))
            all_labels = torch.cat((all_labels, labels))

    # Compute loss
    loss = loss_fn(all_preds, all_labels)

    # Apply sigmoid to predictions
    all_preds = torch.sigmoid(all_preds)

    y_true = all_labels.cpu().numpy()
    y_score = all_preds.cpu().numpy()

    # Original weights (adjust this to your needs)
    class_weights = np.array([1,1,1,1,1,1,1,1,1,1,1,1,1,13], dtype=float)

    # Detect valid classes
    valid_mask = np.array([len(np.unique(y_true[:, j])) == 2 for j in range(y_true.shape[1])])
    valid_classes = np.where(valid_mask)[0].tolist()
    ignored_classes = np.where(~valid_mask)[0].tolist()

    print(f"Ignored classes (only 0s or only 1s): {ignored_classes}")
    print(f"Used classes: {valid_classes}")

    if len(valid_classes) == 0:
        print("No valid classes for ROC AUC.")
        comp_metric = np.nan
    else:
        # Filter inputs and weights
        y_true_valid = y_true[:, valid_classes]
        y_score_valid = y_score[:, valid_classes]
        class_weights_valid = class_weights[valid_classes]

        # Call your existing metric function
        comp_metric = metric.weighted_multilabel_auc(
            y_true_valid, 
            y_score_valid, 
            class_weights_valid
        )

    return loss, comp_metric

    
def train(model, train_loader, optimizer, scheduler, loss_fn, epochs):
    for epoch in range(epochs):
        train_one_epoch(model, train_loader, optimizer, loss_fn)
        scheduler.step()

def set_finetune_params(model, n_top_layers=2):
    # Freeze all backbone layers
    for param in model.backbone.parameters():
        param.requires_grad = False
    # Unfreeze classifier
    for param in model.classifier.parameters():
        param.requires_grad = True
    # Unfreeze last n_top_layers of backbone
    for layer in list(model.backbone.children())[-n_top_layers:]:
        for param in layer.parameters():
            param.requires_grad = True

In [ ]:
# Model

class AneurysmClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = torchvision.models.efficientnet_b3(weights="DEFAULT").features
        self.avgpool = torch.nn.AdaptiveAvgPool2d(1)
        self.classifier = torch.nn.Sequential(
            torch.nn.Dropout(0.3),
            torch.nn.Linear(1536, 256), # 1536 is the number of features for efficientnet_b3
            torch.nn.ReLU(),
            torch.nn.Linear(256, 1)
        )
        self.loc_classifier = torch.nn.Sequential(
            torch.nn.Dropout(0.3),
            torch.nn.Linear(1536, 256), # 1536 is the number of features for efficientnet_b3
            torch.nn.ReLU(),
            torch.nn.Linear(256, 13)
        )
    def forward(self, x):
        # make sure input has 3 channels
        if x.size(1) == 1:
            x = x.repeat(1, 3, 1, 1)
        x = self.backbone(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        loc = self.loc_classifier(x)
        present = self.classifier(x)
        return torch.cat((loc,present), dim=-1)        

In [ ]:
# Loss function
class CustomLoss(torch.nn.Module):
    """
    Compute BCE loss for presence and CE loss for location.
    Args:
        pos_weight (float or tensor): Weight for positive class in BCE loss.
    """
    def __init__(self, pos_weight=None):
        super().__init__()
        self.bce_loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.bce_loss_fn_loc = torch.nn.BCEWithLogitsLoss()

    def forward(self, preds, labels):
        presence_preds, loc_preds = preds[:, -1].unsqueeze(1), preds[:, :-1]
        presence_labels, loc_labels = labels[:, -1].unsqueeze(1), labels[:, :-1]

        presence_loss = self.bce_loss_fn(presence_preds, presence_labels)

        # Only compute location loss for samples with aneurysm present
        if loc_labels.sum() > 0:
            loc_loss = self.bce_loss_fn_loc(loc_preds, loc_labels)
        else:
            loc_loss = torch.tensor(0.0, device=presence_preds.device)

        total_loss = presence_loss + loc_loss
        return total_loss


Hyper parameter search space:
 - `learning_rate`: float, range (1e-5, 1e-2)
 - `loss_pos_weight_alpha`: float, range (0.1, 5.0)
 - `sampling_weight_alpha`: float, range (0.5, 10)
 - `optimizer`: choice of 'adam', 'adamw'
 - `exp_scheduler`: float, range (0.5, 0.99)

In [ ]:
# Optuna hyperparameter search
def objective(trial):
    # Define hyperparameters to tune
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True)
    loss_pos_weight_alpha = trial.suggest_float('loss_pos_weight_alpha', 0.1, 5.0)
    sampling_weight_alpha = trial.suggest_float('sampling_weight_alpha', 0.5, 10.0)
    #optimizer_choice = trial.suggest_categorical('optimizer', ['Adam', 'AdamW'])
    exp_scheduler = trial.suggest_float('exp_scheduler', 0.5, 0.99, log=True)
    
    # Initialize model, optimizer, scheduler, and loss function
    model = AneurysmClassifier()
    model.to(DEVICE)
    pos_weights = torch.tensor((len(negative_indices) / len(positive_indices)) * loss_pos_weight_alpha, dtype=torch.float)
    set_finetune_params(model, n_top_layers=2)
    #optimizer = getattr(torch.optim, optimizer_choice.capitalize())(model.parameters(), lr=learning_rate)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=exp_scheduler)
    loss_fn = CustomLoss(pos_weight=pos_weights)
    val_loss_fn = CustomLoss()
    train_loader = make_train_loader('/kaggle/input/rsna-intracranial-aneurysm-detection', alpha=sampling_weight_alpha, batch_size=32)
    
    # Train the model
    train(model, train_loader, optimizer, scheduler, loss_fn, epochs=5)
    # Test the model on validation set
    val_loader = torch.utils.data.DataLoader(AneurysmDataset('/kaggle/input/rsna-intracranial-aneurysm-detection', train=False, transforms=val_transforms), batch_size=32)
    loss, metric = test_model(model, val_loader, val_loss_fn)
    print(loss, metric)

    if trial.should_prune():
        raise optuna.TrialPruned()
    
    return loss.item()

In [ ]:
#if os.path.exists('/kaggle/input/my-optuna-ds/optuna_study.db'):
import shutil
shutil.copy('/kaggle/input/my-optuna-ds/optuna_study.db', '/kaggle/working/optuna_study.db')
print('reload study (copy in working dir)')

In [ ]:
study = optuna.create_study(study_name='my_study',
                            direction='minimize',
                            pruner=optuna.pruners.SuccessiveHalvingPruner(),
                            sampler=optuna.samplers.TPESampler(),
                            storage='sqlite:////kaggle/working/test.db',
                            load_if_exists=False)
study.optimize(objective, n_trials=10)

print("Number of trials:", len(study.trials))
for t in study.trials:
    print(f"Trial {t.number}: state={t.state}, value={t.value}")

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

In [ ]:
if False:
    #TEST
    model = AneurysmClassifier()
    pos_weights = torch.tensor((len(negative_indices) / len(positive_indices)) * 0.5, dtype=torch.float)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weights)
    optimizer = torch.optim.AdamW(model.parameters())
    dataset = AneurysmDataset('/kaggle/input/rsna-intracranial-aneurysm-detection')
    train(model, dataset, optimizer, loss_fn, epochs = 1)